In [4]:
import csv
import random
import os
from faker import Faker
from datetime import datetime

# Initialize Faker
fake = Faker()
Faker.seed(101)
random.seed(101)

# Configuration
NUM_CUSTOMERS = 600
NUM_PRODUCTS = 600
NUM_ORDERS = 600
NUM_ORDER_ITEMS = 600

# Ensure output directory exists
OUTPUT_DIR = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_customers():
    print(f"Generating {NUM_CUSTOMERS} customers...")
    customers = []
    types = ["REGULAR", "PREMIUM", "VIP"]
    
    for i in range(1, NUM_CUSTOMERS + 1):
        reg_date = fake.date_time_between(start_date="-3y", end_date="now")
        email = fake.email()
        
        # Intentional Issue: 2% of emails should be invalid (missing @ or domain)
        if random.random() < 0.02:
            if random.choice([True, False]):
                email = email.replace("@", "") # Remove @ symbol
            else:
                email = email.split("@")[0] # Remove domain entirely
                
        customers.append({
            "customer_id": f"CUST_{i:04d}",
            "customer_name": fake.name(),
            "email": email,
            "registration_date": reg_date.strftime("%Y-%m-%d %H:%M:%S"),
            "customer_type": random.choices(types, weights=[70, 20, 10])[0]
        })
        
    with open(os.path.join(OUTPUT_DIR, "customers.csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=customers[0].keys())
        writer.writeheader()
        writer.writerows(customers)
    return [c["customer_id"] for c in customers]

def generate_products():
    print(f"Generating {NUM_PRODUCTS} products...")
    products = []
    categories = {
        "Electronics": ["Laptops", "Smartphones", "Accessories", "Audio"],
        "Clothing": ["Men", "Women", "Kids", "Shoes"],
        "Home": ["Furniture", "Decor", "Kitchen", "Bedding"],
        "Books": ["Fiction", "Non-Fiction", "Educational", "Comics"]
    }
    
    for i in range(1, NUM_PRODUCTS + 1):
        cat = random.choice(list(categories.keys()))
        subcat = random.choice(categories[cat])
        product_name = fake.catch_phrase()
        
        # Intentional Issue: Some product names with extra spaces or mixed case (approx 10%)
        if random.random() < 0.10:
            issue_type = random.choice(["extra_spaces", "upper", "mixed"])
            if issue_type == "extra_spaces":
                product_name = f"   {product_name}    "
            elif issue_type == "upper":
                product_name = product_name.upper()
            elif issue_type == "mixed":
                product_name = "".join(random.choice([k.upper(), k.lower()]) for k in product_name)
        
        products.append({
            "product_id": f"PROD_{i:04d}",
            "product_name": product_name,
            "category": cat,
            "subcategory": subcat,
            "cost_price": round(random.uniform(5.0, 500.0), 2)
        })
        
    with open(os.path.join(OUTPUT_DIR, "products.csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=products[0].keys())
        writer.writeheader()
        writer.writerows(products)
    return [p["product_id"] for p in products]

def generate_orders(customer_ids):
    print(f"Generating {NUM_ORDERS} orders...")
    orders = []
    statuses = ["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]
    
    for i in range(1, NUM_ORDERS + 1):
        order_date = fake.date_time_between(start_date="-2y", end_date="now")
        cust_id = random.choice(customer_ids)
        
        # Intentional Issue: 5% of orders should have NULL customer_id
        if random.random() < 0.05:
            cust_id = ""  # Represents NULL in CSV
            
        # Intentional Issue: Some orders have order_date in wrong format (DD-MM-YYYY)
        # Applying this to roughly 5% of the data
        if random.random() < 0.05:
            date_str = order_date.strftime("%d-%m-%Y") 
        else:
            date_str = order_date.strftime("%Y-%m-%d %H:%M:%S")
            
        orders.append({
            "order_id": f"ORD_{i:05d}",
            "customer_id": cust_id,
            "order_date": date_str,
            "status": random.choices(statuses, weights=[10, 20, 50, 10, 10])[0],
            "region_code": fake.state_abbr()
        })
        
    with open(os.path.join(OUTPUT_DIR, "orders.csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=orders[0].keys())
        writer.writeheader()
        writer.writerows(orders)
    return [o["order_id"] for o in orders]

def generate_order_items(order_ids, product_ids):
    print(f"Generating {NUM_ORDER_ITEMS} order items...")
    order_items = []
    
    for i in range(1, NUM_ORDER_ITEMS + 1):
        # Intentional Issue: 3% of order_items should have negative quantity
        if random.random() < 0.03:
            qty = random.randint(-5, -1)
        else:
            qty = random.randint(1, 10)
            
        order_items.append({
            "item_id": f"ITEM_{i:05d}",
            "order_id": random.choice(order_ids), # Guarantees valid FK
            "product_id": random.choice(product_ids), # Guarantees valid FK
            "quantity": qty,
            "unit_price": round(random.uniform(10.0, 800.0), 2),
            "discount_percent": round(random.uniform(0, 100), 2) if random.random() < 0.3 else 0.0 
        })
        
    with open(os.path.join(OUTPUT_DIR, "order_items.csv"), "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=order_items[0].keys())
        writer.writeheader()
        writer.writerows(order_items)

if __name__ == "__main__":
    print("Starting data generation pipeline...")
    # Sequential execution ensures referential integrity
    customer_ids_list = generate_customers()
    product_ids_list = generate_products()
    order_ids_list = generate_orders(customer_ids_list)
    generate_order_items(order_ids_list, product_ids_list)
    
    print(f"Data generation complete! Files saved to '{OUTPUT_DIR}' directory.")

Starting data generation pipeline...
Generating 600 customers...
Generating 600 products...
Generating 600 orders...
Generating 600 order items...
Data generation complete! Files saved to 'data/raw' directory.
